# ChakraModel Eval Harness — Kaggle Runner (v4)

Requires:
1. Dataset **ChakraModel_EvalHarness_Code** (v4 — batch/compile OOM fix) attached, e.g. `/kaggle/input/datasets/jayasrikannuchamy/chakramodel-evalharness-code-new`
2. Dataset **ChakraModel_EvalHarness_Weights** (chakra_transformer_best.pth + combo1_best.pth) attached
3. Any polyp datasets you want scored (PolypGen, HyperKvasir, CVC-ClinicDB, etc.) attached — no code changes needed, `dataset_discovery.py` auto-scans `/kaggle/input`
4. Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (timm needs to fetch the ViT-Large pretrained backbone once before the checkpoint overwrites it)

No patch cells in this version — `MAX_BATCH=8` and `torch.compile(mode="default")` are already fixed at the source in the v4 code zip. Do not add cells that edit `run_eval.py`/`eval_engine.py` after staging; if you need to change batch size again, edit `resource_monitor.py` in the source and re-zip.

In [ ]:
!pip install -q timm albumentations opencv-python-headless psutil

In [ ]:
import os, shutil, glob

WORK = '/kaggle/working'

# --- locate the code dataset. Handles: a literal zip upload, OR Kaggle's
#     auto-extracted layout, under ANY dataset slug (e.g. the -new one). ---
code_zip = None
for p in glob.glob('/kaggle/input/**/ChakraModel_EvalHarness_Code*.zip', recursive=True):
    code_zip = p
    break

if code_zip:
    print('Unpacking code from', code_zip)
    shutil.unpack_archive(code_zip, WORK)
else:
    eh_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == 'eval_harness' and 'run_eval.py' in files:
            eh_dir = root
            break
    assert eh_dir, (
        'Could not find ChakraModel_EvalHarness_Code*.zip or an extracted eval_harness/ '
        'folder anywhere under /kaggle/input — attach the v4 code dataset.'
    )
    dataset_root = os.path.dirname(eh_dir)
    print('Found already-extracted code at', dataset_root, '(Kaggle auto-extracted the zip on upload)')
    shutil.copytree(eh_dir, f'{WORK}/eval_harness', dirs_exist_ok=True)
    src_dir = os.path.join(dataset_root, 'src')
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, f'{WORK}/src', dirs_exist_ok=True)
    print('Copied eval_harness/ and src/ into', WORK)

# --- sanity check: confirm this IS v4 before wasting GPU hours ---
rm_path = f'{WORK}/eval_harness/resource_monitor.py'
ee_path = f'{WORK}/eval_harness/eval_engine.py'
with open(rm_path) as f:
    rm_src = f.read()
with open(ee_path) as f:
    ee_src = f.read()
assert 'MAX_BATCH         = 8' in rm_src, (
    'Staged resource_monitor.py is NOT v4 (MAX_BATCH != 8). '
    'You attached a stale code dataset version — re-upload ChakraModel_EvalHarness_Code_v4.zip.'
)
assert 'mode="default"' in ee_src, (
    'Staged eval_engine.py is NOT v4 (torch.compile mode != "default"). '
    'You attached a stale code dataset version — re-upload ChakraModel_EvalHarness_Code_v4.zip.'
)
print('Verified: staged code is v4 (MAX_BATCH=8, torch.compile mode=default).')

# --- stage weights where run_eval.py's defaults expect them ---
os.makedirs(f'{WORK}/weights/checkpoints', exist_ok=True)
wanted = {'chakra_transformer_best.pth', 'combo1_best.pth'}
found = set()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in wanted and f not in found:
            src = os.path.join(root, f)
            dst = f'{WORK}/weights/checkpoints/{f}'
            shutil.copy(src, dst)
            found.add(f)
            print('Linked', f, 'from', src)
missing = wanted - found
if missing:
    print('WARNING missing weights:', missing, '— attach the ChakraModel_EvalHarness_Weights dataset.')
else:
    print('Both checkpoints staged OK.')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' -', torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), 'CUDA GPU not available — select Accelerator = GPU T4 x2 in notebook settings.'

In [ ]:
%cd /kaggle/working
!python eval_harness/run_eval.py

In [ ]:
# Sanity peek at the results file
import json, glob
res_files = sorted(glob.glob('/kaggle/working/eval_harness/results/evaluation_results_*.json'))
assert res_files, 'No results file produced. Scroll up in the run cell for the traceback.'
with open(res_files[-1]) as f:
    results = json.load(f)
for r in results:
    print(r.get('dataset'), '->', r.get('vit_large', {}).get('dice'), r.get('error', ''))